# Evidence Quality And Controlled Degradation

This notebook validates synthetic, controlled evidence degradation for later research. It does not train a model or implement a Safety Gate.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

from src.data.evidence_quality import run_evidence_quality_experiment

sns.set_theme(style='whitegrid')

## Experimental Assumptions

Evidence quality is represented by completeness, consistency, freshness, and simulated provenance. Applicant selection is fixed-seed and label-blind. The clean data and model dataset remain unchanged; degradation artifacts are separate.

In [ ]:
run_evidence_quality_experiment(
    raw_history_path=PROJECT_ROOT / 'data' / 'raw' / 'synthetic' / 'financial_history.csv',
    model_dataset_path=PROJECT_ROOT / 'data' / 'processed' / 'synthetic' / 'model_dataset.csv',
    output_dir=PROJECT_ROOT / 'experiments' / 'evidence_quality',
    config=None,
)

output_dir = PROJECT_ROOT / 'experiments' / 'evidence_quality'
quality = pd.read_csv(output_dir / 'applicant_evidence_quality.csv')
events = pd.read_csv(output_dir / 'degradation_events.csv')
samples = pd.read_csv(output_dir / 'degraded_evidence_samples.csv')
summary = pd.read_csv(output_dir / 'scenario_summary.csv')
summary

## Clean Versus Degraded Evidence

The event overlay records the simulated defect, field, month, and action. The sample dataset materializes a small set of degraded records for inspection; it deliberately contains no targets, stress probabilities, population labels, or model features.

In [ ]:
events.groupby(['scenario', 'severity', 'simulated_defect_type']).size().rename('event_count').reset_index()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.barplot(data=summary, x='severity', y='mean_completeness_score', hue='scenario', ax=axes[0])
axes[0].set_title('Completeness by Scenario and Severity')
sns.boxplot(data=quality, x='severity', y='missing_month_count', hue='scenario', ax=axes[1])
axes[1].set_title('Missing Monthly Records')
fig.tight_layout()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.barplot(data=summary, x='severity', y='mean_consistency_score', hue='scenario', ax=axes[0])
axes[0].set_title('Consistency Score')
sns.boxplot(data=quality, x='severity', y='consistency_violation_count', hue='scenario', ax=axes[1])
axes[1].set_title('Detected Consistency Violations')
fig.tight_layout()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.boxplot(data=quality, x='severity', y='observation_age_months', hue='scenario', ax=axes[0])
axes[0].set_title('Freshness: Age of Latest Observation')
sns.boxplot(data=quality, x='severity', y='unverified_record_count', hue='scenario', ax=axes[1])
axes[1].set_title('Simulated Unverified Records')
fig.tight_layout()

In [ ]:
dimension_columns = ['completeness_score', 'consistency_score', 'freshness_score', 'provenance_score', 'evidence_quality_score']
quality.groupby(['scenario', 'severity'])[dimension_columns].mean().round(4)

## Limitations

All defects are simulated. The unweighted aggregate quality score is a provisional summary, not a calibrated probability or validated real-world evidence-quality measure. These observations do not establish real-world credit-risk validity or prove a future Safety Gate will improve decisions.